In [1]:
MODE = "train"                    # "test" or "train"
 
DATA_ROOT = "/kaggle/input/datasets/mahimasunilbachhav/amazon-ml-challenge/student_resource/dataset"
WORK = "/kaggle/working"
 
K = 50                           # candidates per dense layer (L1-L3)
ADDR_TOPN = 50                   # candidates from L4
ADDR_CAP = 100                   # address-key buckets larger than this are dropped
MIN_SIM = 0.10                   # dense neighbours below this cosine are discarded
 
DIM_NAME, DIM_COMBO = 128, 256   # SVD sizes for L1 and L2
TFIDF_FIT_ROWS = 1_000_000       # vectoriser vocabulary/IDF learned on this many pool rows
SVD_FIT_ROWS = 300_000           # SVD learned on this many rows
TFIDF_CHUNK = 250_000            # rows transformed per step (bounds memory)
 
EMB_MODEL = "paraphrase-multilingual-MiniLM-L12-v2"   # Apache-2.0, 118M params
EMB_SEQ, EMB_BATCH = 64, 512
ENC_CHUNK = 200_000              # rows encoded per step
 
GENERIC_DF = 0.02                # token in >= 2% of names of its script -> generic
GENERIC_MIN = 50
STRONG_DF = 0.10                 # >= 10% of its script -> legal-suffix level
 
TRAIN_QUERY_BUCKETS = 3000       # train mode: hash buckets 0-2999 of 10000 = 30% of S1
VAL_BUCKETS = 2000               # buckets 0-1999 = validation (same split as the EDA notebook)
 
PAIR_CHUNK = 50_000              # queries per step when merging layers
DELETE_LAYER_CKPT = True         # drop per-layer files once a country's pairs are saved
SEED = 42
 
# Expected minutes per country, from the measured v3 projection (for progress checks).
EXPECTED_MIN = {"test": {"France": 15, "India": 71, "US": 52},
                "train": {"India": 40, "US": 66}}
 
import gc
import hashlib
import heapq
import json
import os
import shutil
import subprocess
import time
from collections import Counter, defaultdict
from pathlib import Path
 
import numpy as np
import pandas as pd
import psutil
import torch
from tqdm.auto import tqdm
 
assert MODE in ("test", "train"), "MODE must be 'test' or 'train'"
assert torch.cuda.is_available(), "GPU not enabled: set Accelerator to GPU T4"
DEV = "cuda"
 
DATA_ROOT = Path(DATA_ROOT)
WORK = Path(WORK)
CKPT = WORK / f"ckpt_{MODE}"
CKPT.mkdir(parents=True, exist_ok=True)
 
T_START = time.time()
REPORT = {"mode": MODE, "settings": {k: v for k, v in globals().items()
                                     if k.isupper() and isinstance(v, (int, float, str))},
          "countries": {}}
 
print("=" * 70)
print(f"MODE        : {MODE}")
print(f"GPU         : {torch.cuda.get_device_name(0)}")
print(f"Checkpoints : {CKPT}")
print("=" * 70)

MODE        : train
GPU         : Tesla T4
Checkpoints : /kaggle/working/ckpt_train


In [2]:
def mem(tag=""):
    """Print RAM (process RSS) and peak GPU memory."""
    rss = psutil.Process().memory_info().rss / 1e9
    free = psutil.virtual_memory().available / 1e9
    gpu = torch.cuda.max_memory_allocated() / 1e9
    print(f"    [MEM] {tag:<26} RAM={rss:5.1f} GB  free={free:5.1f} GB  GPU peak={gpu:4.1f} GB",
          flush=True)
 
 
def purge():
    """Free Python and CUDA memory between stages."""
    gc.collect()
    torch.cuda.empty_cache()
 
 
def log(msg):
    """Timestamped progress line (minutes since the notebook started)."""
    print(f"  [{(time.time()-T_START)/60:6.1f} min] {msg}", flush=True)
 
 
def bar(iterable=None, **kw):
    """tqdm with a refresh interval that keeps committed-run logs readable."""
    kw.setdefault("mininterval", 10)
    return tqdm(iterable, **kw)
 
 
def hash_bucket(eid):
    """Deterministic 0-9999 bucket; identical to the EDA notebook's split."""
    return int(hashlib.md5(eid.encode()).hexdigest(), 16) % 10000

In [3]:
COLS = ["entity_id", "business_name", "business_address", "country"]
 
 
def load_src(path):
    """Read a source TSV. sep='\\t' is mandatory: fields contain commas."""
    return pd.read_csv(path, sep="\t", dtype=str, names=COLS,
                       header=0, keep_default_na=False)
 
 
split = DATA_ROOT / MODE
log(f"loading {MODE} files ...")
s1 = load_src(split / f"{MODE}_source1.tsv")
pool = pd.concat([load_src(split / f"{MODE}_source2.tsv"),
                  load_src(split / f"{MODE}_source3.tsv")], ignore_index=True)
N_S1_FILE = len(s1)
log(f"source1={len(s1):,}   pool (S2+S3)={len(pool):,}")
 
assert pool["entity_id"].str.match(r"^S[23]-").all(), "pool ids must be S2-/S3-"
assert s1["entity_id"].is_unique and pool["entity_id"].is_unique, "duplicate entity ids"
 
GT = {}
if MODE == "train":
    s1["bucket"] = [hash_bucket(e) for e in bar(s1["entity_id"], desc="hash split",
                                                 unit="row", unit_scale=True)]
    s1 = s1[s1["bucket"] < TRAIN_QUERY_BUCKETS].reset_index(drop=True)
    s1["is_val"] = s1["bucket"] < VAL_BUCKETS
    s1[["entity_id", "is_val"]].to_csv(WORK / "train_queries.csv", index=False)
    log(f"train queries: {len(s1):,} ({len(s1)/N_S1_FILE*100:.0f}% of S1), "
        f"of which validation={int(s1['is_val'].sum()):,}")
 
    gt_df = pd.read_csv(split / "train_ground_truth.tsv", sep="\t", dtype=str,
                        names=["source1_entity_id", "matched_entity_ids"],
                        header=0, keep_default_na=False)
    wanted = set(s1["entity_id"])
    for sid, mids in zip(gt_df["source1_entity_id"], gt_df["matched_entity_ids"]):
        if sid in wanted:
            GT[sid] = {m.strip() for m in mids.split(",") if m.strip()}
    del gt_df
    log(f"ground truth loaded for {len(GT):,} queries, "
        f"{sum(len(v) for v in GT.values()):,} true match edges")
 
COUNTRIES = sorted(s1["country"].unique())
for c in COUNTRIES:
    log(f"  {c:<8} queries={int((s1['country']==c).sum()):>9,}   "
        f"pool={int((pool['country']==c).sum()):>10,}")
purge()
mem("after load")

  [   0.0 min] loading train files ...
  [   0.8 min] source1=2,206,821   pool (S2+S3)=10,320,219


hash split:   0%|          | 0.00/2.21M [00:00<?, ?row/s]

  [   1.0 min] train queries: 662,263 (30% of S1), of which validation=441,407
  [   1.1 min] ground truth loaded for 662,263 queries, 2,293,243 true match edges
  [   1.2 min]   India    queries=  265,098   pool= 4,133,346
  [   1.2 min]   US       queries=  397,165   pool= 6,186,873
    [MEM] after load                 RAM=  4.8 GB  free= 27.8 GB  GPU peak= 0.0 GB


In [4]:
LEGAL_RE = (r"\b(llc|l\.l\.c|inc|corp|corporation|ltd|limited|pvt|private|"
            r"llp|lp|pllc|sas|sarl|eurl|sasu|snc|company)\b")
 
ADDR_ABBR = [
    (r"\brd\b", "road"), (r"\bst\b", "street"), (r"\bdr\b", "drive"),
    (r"\bave\b", "avenue"), (r"\bblvd\b", "boulevard"), (r"\bln\b", "lane"),
    (r"\bct\b", "court"), (r"\bcir\b", "circle"), (r"\bhno\b", "house number"),
    (r"\bno\b", "number"), (r"\bopp\b", "opposite"), (r"\bnr\b", "near"),
    (r"\bdist\b", "district"), (r"\bflr\b", "floor"),
]
 
 
def clean_text(s):
    """Shared cleaning. Non-Latin scripts are dropped here; L3 reads raw text."""
    s = s.fillna("").astype(str).str.lower()
    s = (s.str.normalize("NFKD")
          .str.encode("ascii", errors="ignore")
          .str.decode("ascii"))
    s = s.str.replace(r"##\s*", "", regex=True)
    s = s.str.replace(r"\bwww\.", "", regex=True)
    s = s.str.replace(r"\.(com|net|org|co|in|fr|uk|biz|info|io)\b", "", regex=True)
    s = s.str.replace(r"^[\s\-\.\,\!\@\#\$\%\^\&\*\>\<\'\"/\\]+", "", regex=True)
    s = s.str.replace("&", " and ", regex=False)
    s = s.str.replace(r"\bn/?a\b", " ", regex=True)
    s = s.str.replace(r"[^\w\s]", " ", regex=True)
    return s.str.replace(r"\s+", " ", regex=True).str.strip()
 
 
def clean_name(s):
    """Name: shared cleaning + legal suffixes + repeated words (FH FH -> FH)."""
    s = clean_text(s)
    s = s.str.replace(LEGAL_RE, " ", regex=True)
    s = s.str.replace(r"\b(\w+)\s+\1\b", r"\1", regex=True)
    return s.str.replace(r"\s+", " ", regex=True).str.strip()
 
 
def clean_addr(s):
    """Address: shared cleaning + abbreviation expansion."""
    s = clean_text(s)
    for pat, rep in ADDR_ABBR:
        s = s.str.replace(pat, rep, regex=True)
    return s.str.replace(r"\s+", " ", regex=True).str.strip()
 
 
def addr_keys(caddr):
    """
    House number joined to the token after it: '351_hoxie', '742_3'.
    Comma components get reordered between sources, but inside the street
    component the number still precedes the street name.
    """
    toks = caddr.split()
    keys = set()
    for a, b in zip(toks, toks[1:]):
        if a.isdigit():
            a2 = a.lstrip("0")
            b2 = b.lstrip("0") if b.isdigit() else b
            if a2 and b2:
                keys.add(a2 + "_" + b2)
    return tuple(sorted(keys))
 
 
# ---- generic name tokens, learned per script from this mode's pool ---------
PUNCT = ".,()[]{}-'\"&/:;!#@*|"
 
 
def script_of(tok):
    """Unicode block of the first non-Latin char (each Indic script has its
    own 128-codepoint block), or 'latin'."""
    for ch in tok:
        if ord(ch) > 0x024F:
            return ord(ch) >> 7
    return "latin"
 
 
log("learning generic name tokens (per script) ...")
doc_freq, script_names = Counter(), Counter()
for nm in bar(pool["business_name"], desc="token counts", unit="name", unit_scale=True):
    toks = {x for x in (t.strip(PUNCT).lower() for t in nm.split()) if x}
    doc_freq.update(toks)
    script_names.update({script_of(x) for x in toks})
GENERIC = {t for t, c in doc_freq.items()
           if c >= GENERIC_MIN and c >= GENERIC_DF * script_names[script_of(t)]}
STRONG = {t for t in GENERIC if doc_freq[t] >= STRONG_DF * script_names[script_of(t)]}
 
top = sorted(GENERIC, key=lambda t: -doc_freq[t])
log(f"{len(GENERIC)} generic tokens ({len(STRONG)} legal-suffix level). Most frequent:")
print("    " + "  ".join(top[:30]))
nonlatin = [t for t in top if script_of(t) != "latin"]
print(f"    non-Latin ({len(nonlatin)}): " + "  ".join(nonlatin[:25]))
REPORT["generic_top"] = top[:100]
del doc_freq
purge()
 
 
def strip_generic(name):
    """Drop generic tokens; if that empties the name, drop only legal-suffix
    level tokens; if that also empties it, keep the original."""
    toks = name.split()
    norm = [t.strip(PUNCT).lower() for t in toks]
    kept = [t for t, n in zip(toks, norm) if n not in GENERIC]
    if kept:
        return " ".join(kept)
    kept = [t for t, n in zip(toks, norm) if n not in STRONG]
    return " ".join(kept) if kept else name
 
 
def prepare(df):
    """Per-country text columns for all four layers."""
    out = pd.DataFrame({"entity_id": df["entity_id"].values})
    out["cname"] = clean_name(df["business_name"]).values
    out["caddr"] = clean_addr(df["business_address"]).values
    out["combo"] = (out["cname"] + " " + out["caddr"]).str.strip()
    names = [strip_generic(n) for n in df["business_name"]]
    addrs = df["business_address"].tolist()
    out["etext"] = [f"{n} {n} {a}".strip() for n, a in zip(names, addrs)]
    out["akeys"] = [addr_keys(a) for a in out["caddr"]]
    return out

  [   1.2 min] learning generic name tokens (per script) ...


token counts:   0%|          | 0.00/10.3M [00:00<?, ?name/s]

  [   2.0 min] 296 generic tokens (27 legal-suffix level). Most frequent:
    limited  private  llc  ltd  inc  center  pvt  partners  services  लिमिटेड  co  group  corp  प्राइवेट  and  india  holdings  प्रा  लि  లిమిటెడ్  ಲಿಮಿಟೆಡ್  ప్రైవేట్  லிமிடெட்  ಪ್ರೈವೇಟ್  লিমিটেড  பிரைவேட்  প্রাইভেট  લિમિટેડ  પ્રાઇવેટ  ലിമിറ്റഡ്
    non-Latin (281): लिमिटेड  प्राइवेट  प्रा  लि  లిమిటెడ్  ಲಿಮಿಟೆಡ್  ప్రైవేట్  லிமிடெட்  ಪ್ರೈವೇಟ್  লিমিটেড  பிரைவேட்  প্রাইভেট  લિમિટેડ  પ્રાઇવેટ  ലിമിറ്റഡ്  एलएलपी  പ്രൈവറ്റ്  टेक  इंटरनेशनल  ग्लोबल  इंडस्ट्रीज  इंजीनियरिंग  ट्रेडिंग  सॉल्यूशंस  कंस्ट्रक्शंस


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize
from sentence_transformers import SentenceTransformer
 
 
def gpu_topk(Q, P, k=K, q_batch=1024, p_chunk=1_000_000, desc="search"):
    """
    Exact brute-force top-k by inner product on the GPU in fp16.
    Q, P are L2-normalised (inner product = cosine). The full score matrix
    never exists: each pool chunk's top-k is merged into a running top-k.
    Returns I (int32, -1 where score < MIN_SIM) and S (float16), best first.
    """
    k = min(k, P.shape[0])
    P_t = torch.from_numpy(np.ascontiguousarray(P, dtype=np.float16)).to(DEV)
    I = np.empty((Q.shape[0], k), dtype=np.int32)
    S = np.empty((Q.shape[0], k), dtype=np.float16)
    for i in bar(range(0, Q.shape[0], q_batch), desc=desc, unit="batch"):
        q = torch.from_numpy(np.ascontiguousarray(Q[i:i + q_batch], dtype=np.float16)).to(DEV)
        best_v = best_i = None
        for j in range(0, P_t.shape[0], p_chunk):
            s = q @ P_t[j:j + p_chunk].T
            v, ix = torch.topk(s, min(k, s.shape[1]), dim=1)
            ix += j
            if best_v is None:
                best_v, best_i = v, ix
            else:
                cv = torch.cat([best_v, v], dim=1)
                ci = torch.cat([best_i, ix], dim=1)
                best_v, sel = torch.topk(cv, k, dim=1)
                best_i = torch.gather(ci, 1, sel)
            del s
        n = q.shape[0]
        I[i:i + n] = best_i.cpu().numpy()
        S[i:i + n] = best_v.cpu().numpy()
    del P_t
    purge()
    I[S < MIN_SIM] = -1
    return I, S
 
 
def tfidf_svd(q_txt, p_txt, dim, tag):
    """
    Char n-gram TF-IDF (typo tolerant) compressed to `dim` dense dims.
    Vocabulary/IDF and SVD are learned on samples; every row is then
    transformed chunk by chunk, so the full sparse matrix is never held.
    Output is L2-normalised float16.
    """
    rng = np.random.default_rng(SEED)
    fit_idx = rng.choice(len(p_txt), min(TFIDF_FIT_ROWS, len(p_txt)), replace=False)
    t = time.time()
    vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 4), min_df=3,
                          max_df=0.5, max_features=40_000, sublinear_tf=True,
                          dtype=np.float32)
    X = vec.fit_transform([p_txt[i] for i in fit_idx])
    svd_idx = rng.choice(X.shape[0], min(SVD_FIT_ROWS, X.shape[0]), replace=False)
    dim = min(dim, X.shape[1] - 1)
    svd = TruncatedSVD(n_components=dim, algorithm="randomized", n_iter=4,
                       random_state=SEED).fit(X[svd_idx])
    del X
    log(f"{tag}: vectoriser + SVD fitted in {time.time()-t:.0f}s "
        f"(vocab {len(vec.vocabulary_):,}, explained-var {svd.explained_variance_ratio_.sum():.2f})")
 
    def transform(texts, desc):
        out = np.empty((len(texts), dim), dtype=np.float16)
        for i in bar(range(0, len(texts), TFIDF_CHUNK), desc=desc, unit="chunk"):
            out[i:i + TFIDF_CHUNK] = normalize(svd.transform(vec.transform(texts[i:i + TFIDF_CHUNK])))
        return out
 
    P = transform(p_txt, f"{tag} pool")
    Q = transform(q_txt, f"{tag} queries")
    del vec, svd
    purge()
    return Q, P
 
 
def encode(texts, desc):
    """Encode raw text in chunks. Raw text keeps every Indian script."""
    out = np.empty((len(texts), EMB_DIM), dtype=np.float16)
    pb = bar(total=len(texts), desc=desc, unit="row", unit_scale=True)
    for i in range(0, len(texts), ENC_CHUNK):
        part = model.encode(texts[i:i + ENC_CHUNK], batch_size=EMB_BATCH,
                            normalize_embeddings=True, convert_to_numpy=True,
                            show_progress_bar=False)
        out[i:i + len(part)] = part
        pb.update(len(part))
    pb.close()
    return out
 
 
def addr_layer(q_keys, p_keys):
    """L4: pool rows sharing address keys, ranked by number of shared keys
    (ties broken by pool index, so results are reproducible run to run).
    Returns I (int32, -1 padded) and C (int8 shared-key counts)."""
    buckets = defaultdict(list)
    for j, ks in enumerate(bar(p_keys, desc="L4 index", unit="row", unit_scale=True)):
        for key in ks:
            buckets[key].append(j)
    for key in [k for k, v in buckets.items() if len(v) > ADDR_CAP]:
        del buckets[key]
    I = np.full((len(q_keys), ADDR_TOPN), -1, dtype=np.int32)
    C = np.zeros((len(q_keys), ADDR_TOPN), dtype=np.int8)
    for i, ks in enumerate(bar(q_keys, desc="L4 query", unit="row", unit_scale=True)):
        if not ks:
            continue
        cnt = Counter()
        for key in ks:
            b = buckets.get(key)
            if b:
                cnt.update(b)
        if cnt:
            best = heapq.nsmallest(ADDR_TOPN, cnt.items(), key=lambda kv: (-kv[1], kv[0]))
            I[i, :len(best)] = [b[0] for b in best]
            C[i, :len(best)] = [min(b[1], 127) for b in best]
    del buckets
    purge()
    return I, C
 
 
def build_pairs(layers, IA, CA, n_pool):
    """
    Union of the four layers as flat arrays, one row per (query, candidate).
    l1/l2/l3 hold each layer's cosine (0 = not retrieved by that layer) and ak
    the shared address-key count. Rows come out sorted by query, then pool index.
    """
    nq = IA.shape[0]
    parts = defaultdict(list)
    for r0 in bar(range(0, nq, PAIR_CHUNK), desc="union", unit="chunk"):
        r1 = min(nq, r0 + PAIR_CHUNK)
        blocks_i = [I[r0:r1] for I, _ in layers] + [IA[r0:r1]]
        blocks_v = [S[r0:r1].astype(np.float32) for _, S in layers] + [CA[r0:r1].astype(np.float32)]
        widths = [b.shape[1] for b in blocks_i]
        Pm = np.concatenate(blocks_i, axis=1)
        Vm = np.concatenate(blocks_v, axis=1)
        Lm = np.broadcast_to(np.repeat(np.arange(4, dtype=np.int64), widths), Pm.shape)
        Qm = np.broadcast_to(np.arange(r0, r1, dtype=np.int64)[:, None], Pm.shape)
        valid = Pm >= 0
        keys = Qm[valid] * n_pool + Pm[valid].astype(np.int64)
        uk, inv = np.unique(keys, return_inverse=True)
        feats = np.zeros((4, len(uk)), dtype=np.float32)
        feats[Lm[valid], inv] = Vm[valid]     # a layer never repeats a (q, p) pair
        parts["q"].append((uk // n_pool).astype(np.int32))
        parts["p"].append((uk % n_pool).astype(np.int32))
        for li, name in enumerate(("l1", "l2", "l3")):
            parts[name].append(feats[li].astype(np.float16))
        parts["ak"].append(feats[3].astype(np.int8))
    return {k: np.concatenate(v) for k, v in parts.items()}
 
 
def label_pairs(pairs, qids, pids, n_pool):
    """Train mode: 1 where the pair is a true match. Also returns true-edge
    counts; a true match missing from this country's pool counts as missed."""
    qpos = {sid: i for i, sid in enumerate(qids)}
    ppos = {pid: j for j, pid in enumerate(pids)}
    true_keys, n_true = [], 0
    for sid in qids:
        for m in GT.get(sid, ()):
            n_true += 1
            j = ppos.get(m)
            if j is not None:
                true_keys.append(qpos[sid] * n_pool + j)
    del ppos
    true_keys = np.unique(np.array(true_keys, dtype=np.int64))
    keys = pairs["q"].astype(np.int64) * n_pool + pairs["p"].astype(np.int64)
    return np.isin(keys, true_keys).astype(np.int8), n_true
 
 
log(f"loading {EMB_MODEL} (fp16, max {EMB_SEQ} tokens) ...")
model = SentenceTransformer(EMB_MODEL, device=DEV)
model.max_seq_length = EMB_SEQ
model.half()
EMB_DIM = model.get_sentence_embedding_dimension()
mem("after model load")

  [   2.5 min] loading paraphrase-multilingual-MiniLM-L12-v2 (fp16, max 64 tokens) ...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

    [MEM] after model load           RAM=  5.8 GB  free= 27.1 GB  GPU peak= 0.7 GB


/tmp/ipykernel_23/1394109008.py:172: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  EMB_DIM = model.get_sentence_embedding_dimension()


In [6]:
def run_layer(path, fn):
    """Load a finished layer from disk, or compute and save it."""
    if path.exists():
        d = np.load(path)
        log(f"{path.stem}: loaded from checkpoint")
        return d["I"], d["V"]
    I, V = fn()
    np.savez(path, I=I, V=V)
    return I, V
 
 
for country in COUNTRIES:
    cdir = CKPT / country
    cdir.mkdir(exist_ok=True)
    if (cdir / "pairs.npz").exists():
        log(f"[{country}] already complete — skipping")
        continue
 
    t_c = time.time()
    print("\n" + "=" * 70)
    exp = EXPECTED_MIN.get(MODE, {}).get(country)
    print(f"COUNTRY {country}" + (f"   (expected ~{exp} min)" if exp else ""))
    print("=" * 70, flush=True)
 
    q_raw = s1[s1["country"] == country]
    p_raw = pool[pool["country"] == country]
    if len(p_raw) == 0:
        log("no pool rows for this country — all queries get empty candidates")
        empty = {k: np.zeros(0, dt) for k, dt in
                 (("q", np.int32), ("p", np.int32), ("l1", np.float16),
                  ("l2", np.float16), ("l3", np.float16), ("ak", np.int8))}
        np.save(cdir / "qids.npy", q_raw["entity_id"].to_numpy(dtype=object), allow_pickle=True)
        np.save(cdir / "pids.npy", np.zeros(0, dtype=object), allow_pickle=True)
        np.savez(cdir / "pairs.npz", **empty)
        continue
 
    t = time.time()
    q = prepare(q_raw)
    p = prepare(p_raw)
    del q_raw, p_raw
    qids = q["entity_id"].to_numpy(dtype=object)
    pids = p["entity_id"].to_numpy(dtype=object)
    np.save(cdir / "qids.npy", qids, allow_pickle=True)
    np.save(cdir / "pids.npy", pids, allow_pickle=True)
    n_pool = len(p)
    log(f"prepared {len(q):,} queries / {n_pool:,} pool rows in {time.time()-t:.0f}s")
    mem("after prepare")
 
    # L1 — name TF-IDF
    def _l1():
        Q, P = tfidf_svd(q["cname"].tolist(), p["cname"].tolist(), DIM_NAME, "L1 name")
        return gpu_topk(Q, P, desc="L1 search")
    I1, S1 = run_layer(cdir / "L1.npz", _l1)
    purge(); mem("after L1")
 
    # L2 — name+address TF-IDF
    def _l2():
        Q, P = tfidf_svd(q["combo"].tolist(), p["combo"].tolist(), DIM_COMBO, "L2 combo")
        return gpu_topk(Q, P, desc="L2 search")
    I2, S2 = run_layer(cdir / "L2.npz", _l2)
    purge(); mem("after L2")
 
    # L3 — multilingual embedding
    def _l3():
        Pe = encode(p["etext"].tolist(), "L3 encode pool")
        Qe = encode(q["etext"].tolist(), "L3 encode queries")
        return gpu_topk(Qe, Pe, desc="L3 search")
    I3, S3 = run_layer(cdir / "L3.npz", _l3)
    purge(); mem("after L3")
 
    # L4 — address keys
    IA, CA = run_layer(cdir / "L4.npz",
                       lambda: addr_layer(q["akeys"].tolist(), p["akeys"].tolist()))
    del q, p
    purge(); mem("after L4")
 
    # Union of all layers
    pairs = build_pairs([(I1, S1), (I2, S2), (I3, S3)], IA, CA, n_pool)
    del I1, S1, I2, S2, I3, S3, IA, CA
    purge()
 
    stats = {"queries": int(len(qids)), "pool": int(n_pool), "pairs": int(len(pairs["q"]))}
    if MODE == "train":
        pairs["label"], n_true = label_pairs(pairs, qids, pids, n_pool)
        is_val = s1.set_index("entity_id").loc[list(qids), "is_val"].to_numpy()
        val_q = is_val[pairs["q"]]
        lab = pairs["label"].astype(bool)
        n_true_val = sum(len(GT.get(sid, ())) for sid, v in zip(qids, is_val) if v)
        stats.update({
            "true_edges": n_true, "captured": int(lab.sum()),
            "true_edges_val": n_true_val, "captured_val": int((lab & val_q).sum()),
            "captured_by_layer": {name: int((lab & (pairs[name] > 0)).sum())
                                  for name in ("l1", "l2", "l3", "ak")},
        })
    np.savez(cdir / "pairs.npz", **pairs)
    with open(cdir / "stats.json", "w") as f:
        json.dump(stats, f, indent=2)
 
    if DELETE_LAYER_CKPT:
        for name in ("L1", "L2", "L3", "L4"):
            (cdir / f"{name}.npz").unlink(missing_ok=True)
 
    mins = (time.time() - t_c) / 60
    avg_c = stats["pairs"] / max(stats["queries"], 1)
    msg = f"[{country}] DONE in {mins:.1f} min   pairs={stats['pairs']:,}   avg candidates={avg_c:.0f}"
    if MODE == "train" and stats["true_edges"]:
        msg += f"   recall={stats['captured']/stats['true_edges']*100:.2f}%"
    log(msg)
    del pairs
    purge()
    mem("country end")
 
log("all countries complete")


COUNTRY India   (expected ~40 min)
  [   6.7 min] prepared 265,098 queries / 4,133,346 pool rows in 233s
    [MEM] after prepare              RAM=  9.6 GB  free= 23.2 GB  GPU peak= 0.7 GB
  [   7.1 min] L1 name: vectoriser + SVD fitted in 27s (vocab 40,000, explained-var 0.40)


L1 name pool:   0%|          | 0/17 [00:00<?, ?chunk/s]

L1 name queries:   0%|          | 0/2 [00:00<?, ?chunk/s]

L1 search:   0%|          | 0/259 [00:00<?, ?batch/s]

    [MEM] after L1                   RAM=  9.6 GB  free= 23.2 GB  GPU peak= 3.4 GB
  [  12.2 min] L2 combo: vectoriser + SVD fitted in 110s (vocab 40,000, explained-var 0.39)


...

L4 index:   0%|          | 0.00/6.19M [00:00<?, ?row/s]

L4 query:   0%|          | 0.00/397k [00:00<?, ?row/s]

    [MEM] after L4                   RAM=  8.0 GB  free= 24.9 GB  GPU peak= 7.1 GB


union:   0%|          | 0/8 [00:00<?, ?chunk/s]

  [  98.3 min] [US] DONE in 56.4 min   pairs=56,291,083   avg candidates=142   recall=97.35%
    [MEM] country end                RAM=  7.5 GB  free= 25.4 GB  GPU peak= 7.1 GB
  [  98.3 min] all countries complete


In [7]:
def load_country(c):
    d = np.load(CKPT / c / "pairs.npz")
    qids = np.load(CKPT / c / "qids.npy", allow_pickle=True)
    pids = np.load(CKPT / c / "pids.npy", allow_pickle=True)
    stats_path = CKPT / c / "stats.json"
    stats = json.load(open(stats_path)) if stats_path.exists() else {}
    return d, qids, pids, stats
 
 
total_pairs, zero_rows, rows_written = 0, 0, 0
for c in COUNTRIES:
    d, qids, pids, stats = load_country(c)
    total_pairs += len(d["q"])
    REPORT["countries"][c] = stats
 
if MODE == "test":
    out_path = WORK / "candidate_pairs.tsv"
    log(f"writing {out_path.name} ({total_pairs:,} pairs) ...")
    seen = set()
    with open(out_path, "w", encoding="utf-8") as f:
        f.write("source1_entity_id\tcandidate_entity_ids\n")
        for c in COUNTRIES:
            d, qids, pids, _ = load_country(c)
            qa, pa = d["q"], d["p"]
            bounds = np.searchsorted(qa, np.arange(len(qids) + 1))
            for i in bar(range(len(qids)), desc=f"write {c}", unit="row", unit_scale=True):
                lo, hi = bounds[i], bounds[i + 1]
                if lo == hi:
                    zero_rows += 1
                f.write(f"{qids[i]}\t{','.join(pids[pa[lo:hi]])}\n")
                seen.add(qids[i])
                rows_written += 1
    log(f"written: {out_path}  ({out_path.stat().st_size/1e9:.2f} GB)")
 
    # Format checks against the official rules.
    checks = [
        ("one row per test S1 entity", rows_written == N_S1_FILE and seen == set(s1["entity_id"])),
        ("no duplicate S1 rows", len(seen) == rows_written),
        ("candidates are S2-/S3- ids (pool asserted at load)", True),
        ("no duplicate ids within a row (union is deduplicated)", True),
    ]
    print()
    for desc, ok in checks:
        print(f"  {'PASS' if ok else 'FAIL'}  {desc}")
 
    # Official validator, run with a placeholder matching file (empty matches).
    validator = DATA_ROOT.parent / "utils" / "validate_submission.py"
    if validator.exists():
        tmp_match = WORK / "_tmp_empty_matching.tsv"
        with open(tmp_match, "w", encoding="utf-8") as f:
            f.write("source1_entity_id\tmatched_entity_ids\n")
            for sid in s1["entity_id"]:
                f.write(f"{sid}\t\n")
        log("running the official validator on candidate_pairs.tsv ...")
        try:
            r = subprocess.run(["python3", str(validator), "--matching", str(tmp_match),
                                "--candidate", str(out_path), "--test-dir", str(DATA_ROOT / "test")],
                               capture_output=True, text=True, timeout=3600)
            print(r.stdout[-3000:], r.stderr[-2000:])
            REPORT["validator_exit_code"] = r.returncode
            print("  VALIDATOR:", "PASS" if r.returncode == 0 else "FAILED — read the messages above")
        except Exception as e:
            print(f"  validator could not run: {e}")
        tmp_match.unlink(missing_ok=True)
    else:
        print(f"  validator not found at {validator} — skipped")
 
else:
    print("\n" + "=" * 70)
    print("BLOCKING RECALL  (train queries vs the full train pool)")
    print("=" * 70)
    tot = Counter()
    for c in COUNTRIES:
        st = REPORT["countries"][c]
        for k in ("true_edges", "captured", "true_edges_val", "captured_val"):
            tot[k] += st[k]
        for k, v in st["captured_by_layer"].items():
            tot["layer_" + k] += v
        print(f"  {c:<8} recall {st['captured']/st['true_edges']*100:6.2f}%   "
              f"({st['captured']:,}/{st['true_edges']:,})   "
              f"avg candidates {st['pairs']/st['queries']:.0f}")
    recall = tot["captured"] / tot["true_edges"]
    recall_val = tot["captured_val"] / tot["true_edges_val"]
    print(f"\n  OVERALL recall           {recall*100:6.2f}%")
    print(f"  VALIDATION-split recall  {recall_val*100:6.2f}%   (use this in the write-up)")
    print("  recall of each layer alone:")
    for k, name in (("l1", "L1 name TF-IDF"), ("l2", "L2 name+addr TF-IDF"),
                    ("l3", "L3 embedding"), ("ak", "L4 address keys")):
        print(f"    {name:<22} {tot['layer_'+k]/tot['true_edges']*100:6.2f}%")
    REPORT.update({"recall": recall, "recall_val": recall_val})
    print("\n  verification sample projected ~95%; "
          + ("on track." if recall >= 0.92 else "lower than projected — share this block."))


BLOCKING RECALL  (train queries vs the full train pool)
  India    recall  90.01%   (827,206/919,062)   avg candidates 152
  US       recall  97.35%   (1,337,784/1,374,181)   avg candidates 142

  OVERALL recall            94.41%
  VALIDATION-split recall   94.41%   (use this in the write-up)
  recall of each layer alone:
    L1 name TF-IDF          56.59%
    L2 name+addr TF-IDF     75.04%
    L3 embedding            70.60%
    L4 address keys         60.60%

  verification sample projected ~95%; on track.


In [8]:
n_q = sum(REPORT["countries"][c].get("queries", 0) for c in COUNTRIES)
REPORT["total_pairs"] = int(total_pairs)
REPORT["avg_candidates"] = total_pairs / max(n_q, 1)
REPORT["runtime_min"] = (time.time() - T_START) / 60
with open(WORK / f"blocking_report_{MODE}.json", "w") as f:
    json.dump(REPORT, f, indent=2, default=str)
 
print("\n" + "=" * 70)
print(f"NOTEBOOK 2 COMPLETE — MODE={MODE}")
print("=" * 70)
print(f"  queries          : {n_q:,}")
print(f"  candidate pairs  : {total_pairs:,}")
print(f"  avg per query    : {REPORT['avg_candidates']:.0f}")
if MODE == "test":
    print(f"  queries with none: {zero_rows:,}")
print(f"  runtime          : {REPORT['runtime_min']:.0f} min")
print(f"  report           : {WORK / f'blocking_report_{MODE}.json'}")
print("  Stage 2 inputs   : ckpt_{0}/<country>/pairs.npz + qids.npy + pids.npy".format(MODE))
mem("final")


NOTEBOOK 2 COMPLETE — MODE=train
  queries          : 662,263
  candidate pairs  : 96,484,876
  avg per query    : 146
  runtime          : 98 min
  report           : /kaggle/working/blocking_report_train.json
  Stage 2 inputs   : ckpt_train/<country>/pairs.npz + qids.npy + pids.npy
    [MEM] final                      RAM=  7.6 GB  free= 25.1 GB  GPU peak= 7.1 GB
